# Lesson 5: Time Series Data with Datetime and Pandas

In this lesson, we will work with time series data using the datetime and pandas Python libraries.

Time series data refers to data whose primary axis (or dimension) is time. In-situ monitoring instruments are a classic example of time series data, collecting one or more environmental variables at regular time intervals. 

There are many ways to work with time series data in Python. In this lesson, we will first explore the datetime objects available from the datetime library, that form the foundation of working with time related data. Then, we will explore how we can use pandas (recall our friend for reading and analyzing tabular data) to easily read in existing time series data, perform temporal processing (e.g., aggregations along the time axis), and quickly create time series visualizations.

### Learning outcomes

After this lesson, you should be able to:

1) Create date, time, and datetime objects from scratch.
2) Convert between strings and datetime objects.
3) Filter time-series data based on datetimes using a pandas datetime index.
4) Resample time-series data to different temporal frequencies using pandas.
5) Recall the difference between resample and groupby when working with time-series data.

:::{danger}
## Entry ticket!
Before we start, discuss with your neighbor the concept you found the most challenging from the last lesson.
:::

## Real-world time series data

To begin exploring time series data in Python, we will continue working with the USGS stream gauge data from our first lesson on tabular data and pandas. You can view the original data at: https://waterdata.usgs.gov/monitoring-location/USGS-06711565/. Note that the Excel data we will be working with was modified slighlty from the original website output.

In [ ]:
# To open excel data, we'll first need to install an extra package dependency
!pip install openpyxl -qq

# Import the libary - you only have to do this once per file
import pandas as pd

# These settings optional change the way in which pandas data is displayed in the notebook
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 15)

In [ ]:
# Read in the data - this time specify correct filepath
filepath = './data/usgs_streamgauge_06711565.xlsx' # this time we fixed the relative path
gauge = pd.read_excel(filepath)
gauge.head(3) # show the first three rows

In [ ]:
# Data cleaning and preparation

# Show the shape of our dataframe we start with
print('Initial shape:', gauge.shape) # (rows, columns)

# Remove any columns that only contain "NaN" (missing values)
gauge = gauge.dropna(axis=1, how='all')

# Keep only columns we are interested in - see how we provide a list of the column names we want to keep, inside of the [] brackets
gauge = gauge[['time', 'T_degC', 'Q_ft3s-1', 'approval_status']]

# Rename the environmental variable columns just so quicker to type and work with
new_colnames = {'T_degC':'Temperature', 
                 'Q_ft3s-1':'Discharge'} # notice this is a dictionary
gauge = gauge.rename(columns=new_colnames)

# Ensure temp and discharge columns are float values
gauge[['Temperature', 'Discharge']] = gauge[['Temperature', 'Discharge']].astype(float)

# Remove any rows where either temp or discharge is missing value
gauge = gauge.dropna(subset=['Temperature', 'Discharge'], how='any')

# Sort from oldest to newest, reset the index row integers starting from 0
gauge = gauge.sort_values('time')
gauge = gauge.reset_index(drop=True)

# Look how the shape of our dataframe changed - 
# .shape and .size are great functions to keep track of what's happening overall to your data during cleaning/preparation
print('Final shape:', gauge.shape) 
print('Final size:', gauge.size) 

gauge

## Datetimes

Hopefully you already noticed that our gauge data has time information. If you look closely, you'll notice that each row represents a 15 minute time interval. By default, our time column is simply represented by a string object. However, dates and times have special properties. For this reaosn, it is often beneficial to represent these dates and times with specific data types that "unlock" new functionality. 

There is a whole `datetime` library dedicated to this and it contains many helpful functions to make our lives easier working with dates and times. The `datetime` library has three major components. Let's briefly practice some basic examples before we move ahead with our real-world data.
* `date()`: for handling a date (e.g., 2021-06-28)
* `datetime()`: for handling a specific time of a particular date (e.g., 2021-06-28 at 12:31pm)
* `timedelta()`: for handling a length of time (e.g., 1 hour, 2 weeks, 3.5 years)

In [ ]:
# We'll import specific classes from the datetime library
from datetime import date, datetime, timedelta

# assigning a date to a variable
birthday = date(2019, 9, 12)
print(birthday)
birthday

In [ ]:
# Use .hour attribute of a datetime
example = datetime(2021, 9, 21, 13, 45, 0)
if example.hour >= 12 and example.hour <= 18:
    print(example, 'sample taken in the afternoon')

In [ ]:
# Calculate a timedelta simply using subtraction
# first convert datetime to date for consistency with birthday
time_diff = example.date() - birthday
print('difference', time_diff)
print('difference as seconds:', time_diff.total_seconds())

In [ ]:
# We can also create timedelta manually using timedelta()
# Representing 3 days
print('3 days from today', date.today() + timedelta(days=3))

# Negatives also work
print('3 days before today', date.today() + timedelta(days=-3))

:::{admonition}
### Exercise: Datetime objects

1. Create a `date` object for your birthday. If you know the hour (and minutes) you were born, you can create a `datetime` object instead.
2. Create a `timedelta` object as the difference between today and your birthday.
3. Create a dictionary that contains `date` objects for your three favorite national holidays. For each holiday, put the name of the holiday as the key and the `date` object as the value. Use 2025 as the year.

:::

## Converting between strings and datetimes

So far we have created our datetime objects by inputing the integer number of the year, month, etc.  Much more commonly, though, we will be creating datetime objects them from a string.  For that we have `strptime` and `strftime`:

| Syntax      | Description |
| :----------- | :----------- |
| `datetime.strptime()`      | Convert from string to datetime       |
| `DATETIME_OBJECT.strftime()`   | Convert from datetime to string       |

All of the following strings can be converted programatically into datetime objects, we just need to ensure to provide the correct corresponding datetime format:
* `'2019-01-29'`
* `'01-31-88'`
* `'September 12 7:00PM'`
* `'Apr 11 2001 13:00:00'`
* `'210 9:00'`
* Etc.

There are a couple of handy cheatsheets and tools to help decipher the right string format: 
* https://www.bairesdev.com/tools/strftime/
* https://www.pythonmorsels.com/strptime/

In [ ]:
# Let's test out strptime to convert from string to datetime object
datetime.strptime('2019-01-29', "%y-%m-%d")

:::{warning}
## Bug alert!

`ValueError: time data '2019-01-29' does not match format '%y-%m-%d'`

The date format we gave was incorrect for the date string.

In [ ]:
# Try again with correct date format - %Y, not %y
datetime.strptime('2019-01-29', "%Y-%m-%d")

In [ ]:
datetime.strptime('Apr 11 2001 13:00:00', '%b %d %Y %H:%M:%S')

:::{admonition}
### Exercise: Datetime string formats

What is the correct string format you would use to convert each of these strings to a datetime object? If you are struggling, you may use this cheatsheet to help you: https://www.bairesdev.com/tools/strftime/

1) `'2026-148'`

a) `"%m/%d/%Y %I:%M:%S %p"`

b) `"%Y-%m-%dT%H:%M:%S"`

c) `"%Y-%j"`

2) `'2019-01-29T08:47:23'`

a) `"%m/%d/%Y %I:%M:%S %p"`

b) `"%Y-%m-%dT%H:%M:%S"`

c) `"%Y-%j"`

Bonus: for the answer you haven't selected yet (a, b, or c), provide an example datetime string in this format.

:::

## Datetimes and pandas: the best of friends

Fortunately, pandas (our old friend for working with tabular data) provides many neat functions that are compatible with `datetime` and make processing and analyzing time series data much easier.

With this new knowledge of datetime objects, let's get back to our real-world streamgauge data. Here, we'll use the function `pd.to_datetime()` to **convert the time strings into datetime objects**. Fortunately, `pd.to_datetime()` is smart enough to automatically detect the string date format, meaning we won't usually need to manually specify the format like we did earlier. That being said, it is always good to check for yourself that the default output is consistent with what you expect it to be.

Once we have converted our time column to datetime objects, we can take this one step further and **set this newly formatted time column as the index of our dataframe.** We will do this using the pandas function `set_index()`. These steps combined open up a whole world of analysis and visualization options, which we'll briefl explore here.

In [ ]:
print(type(gauge.time.iloc[0]))

gauge['time'] = pd.to_datetime(gauge['time'])
# Manually specifying the correct string format will give the same answer
# gauge['time'] = pd.to_datetime(gauge['time'], format="%Y-%m-%d %H:%M:%S%z")

# Notice we have converted from a simple string to a pandas timestamp 
print(type(gauge.time.iloc[0]))

gauge

In [ ]:
# Set time as index and create simple time-series plot
gauge.set_index('time', inplace=True)
# note that using inpalce=True is same as using 
# gauge = gauge.set_index('time')
gauge

In [ ]:
# Sort by temperature (or any other column) using sort_values()
gauge.sort_values(by='Temperature', inplace=True, ascending=False) # sort from highest to lowest
gauge

In [ ]:
# Sort by time index using sort_index
gauge.sort_index(inplace=True, ascending=True) # sorts in order from oldest to newest
gauge

In [ ]:
# Quick plot - note how the x-axis are automatically formatted as dates
# Compare this with the plot we made in lesson 2 tabular data using the same data
gauge.plot()

## Date-based indexing and filtering

Recall that we can use comparison operators (`==`, `!=`, `>`, etc.) to filter our dataframes based on conditions. Having set up our datetime index, we can now apply this same filtering concept using datetimes conditions. Since we are now indexing values based on their index labels/names (datetimes), we need to use `loc` instead of `iloc`.

In [ ]:
# Filter for specific date window
gauge_window = gauge.loc['2025-05':'2025-10']

# Plot time-series of discharge
gauge_window['Discharge'].plot()

:::{admonition}
### Exercise: Datetime filtering practice

Can you write a line of code to do each of the following:

1. Filter for only data from 2025.

2. Filter for data between June 1 and June 21, 2025. Plot your resulting temperature data during this date window.

3. Find the most frequent discharge value that occurred in 2026 and the number of times it occured. (Hint: recall the pandas function `value_counts()` from our tabular data lesson.)

:::

:::{dropdown}
### Solution

1. `gauge.loc['2025']`

2. `gauge.loc['2025-06-01':'2025-06-21', 'Temperature'].plot()` or `gauge.loc['2025-06-01':'2025-06-21']['Temperature'].plot()`

3. `gauge.loc['2026', 'Discharge'].value_counts()` or  `gauge.loc['2026']['Discharge'].value_counts()`
:::

## Temporal resampling

One of the common operations when working with time series data is resampling. Imagine you have _daily_ data but what you want is a time series plot showing mean value for each _month_. Pandas provides a helpful function `resample()` to deal with these instances (see the [docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html)). 

The main argument required by `resample()` is the temporal interval over which we want to resample. In pandas, this temporal interval is called the frequency. As such, we need the corresponding frequency string indicator (for example, daily uses `'D'`). The other element we need is the aggregation operation to apply over the frequency interval. Common resampling aggregations include: `mean()`, `min()`, `max()`.

Let's try out some resampling using our stream gauge data. Recall that we have data at 15 minute intervals. Let's convert this to hourly maximum data.

In [ ]:
gauge
print(gauge.shape)

# Resample to daily, taking the max values on each day
gauge_daily_max = gauge.resample("D").max()

# Note we now have many fewer rows
print(gauge_daily_max.shape)

gauge_daily_max

In [ ]:
# We can check the result of the resampling by inspecting our new time index
# Note the freq='D' telling us this datetime index has daily frequency
gauge_daily_max.index

In [ ]:
# Plotting our daily max data
gauge_daily_max['Discharge'].plot(marker='.')

:::{admonition}
### Exercise: Temporal resampling

Let's test out some other options for temporal resampling.

1. Using the original `gauge` dataframe, create a new dataframe containing only the discharge column. (Hint: use the double `[[]]` syntax to create a dataframe rather than a dataseries, i.e., `gauge[['Discharge']]`). Resample the original 15-minute gauge data to _hourly_ frequency, taking the _maximum_ in each hour. Then, filter for a single day (you can pick any day from our data that you wish). Plot the resampled hourly maximum discharge data on your chosen day.

2. Using the original `gauge` dataframe, create a new dataframe containing only the temperature column. Resample the original 15-minute data to _monthly_ frequency, taking the _mean_ in each month. Plot the resampled temperature data.

:::

In [ ]:
discharge = gauge[['Discharge']]
discharge_hourly_max = discharge.resample('h').max()
discharge_hourly_max_20250601 = discharge_hourly_max.loc['2025-06-01']
discharge_hourly_max_20250601['Discharge'].plot(marker='.')

In [ ]:
temp = gauge[['Temperature']]
temp_monthly_mean = temp.resample('ME').mean()
temp_monthly_mean['Temperature'].plot(marker='.')

:::{dropdown}
### Solution

1. 

```
discharge = gauge[['Discharge']]
discharge_hourly_max = discharge.resample('h').max()
discharge_hourly_max_20250601 = discharge_hourly_max.loc['2025-06-01']
discharge_hourly_max_20250601['Discharge'].plot(marker='.')
```

2.

```
temp = gauge[['Temperature']]
temp_monthly_mean = temp.resample('ME').mean()
temp_monthly_mean['Temperature'].plot(marker='.')
```
:::

:::{danger}
## Exit ticket!

Before you leave this lesson, please [submit your responses here](https://forms.gle/cVjyNwBZyUycm6RT6) to three quick questions: 

* How much of this lesson's content do you feel comfortable with? 0-10, with 10 being all of the content.
* How was the pace of this lesson for you? 1) Too slow; 2) About right; 3) Too fast.
* In a few words or less, what single concept was most challenging for you in this lesson?

:::

:::{hint}
## Bonus concept: Groupby for creating climatologies

In this lesson we resampled our stream gauge time seried data to different frequency time intervals. The result of our monthly resampling for example gave us the mean value for each month in our data. But what if we had many years of data and wanted just the average for each month over the entire time period of the data? This is what we call a climatology: the average conditions (e.g., in a specific month or season) over a multi-year historical period.

To do this type of operation, we need to move from `resample()` to a slightly different pandas function: `groupby()`. Just like `resample()`, `groupby()` also requires two main things: the interval (or group) to aggregate over, and the aggregation function to apply (min, max, mean, etc.) 

To calculate the monthly climatology in our example, we would use:

```
gauge.groupby(by=gauge.index.month).mean()
```

Try it for yourself and see how the output differs from when we resampled to monthly frequency!

As matter of fact, `groupby()` works on all types of data, not just time. In this sense, it is a more flexible function than resample which is specific to time-series data. For example, if you had data containing multiple entries for each county and you wanted to calculate aggregate statistics per county, you would use `groupby(by='county')` followed by whatever aggregation function you want (e.g., `.max()`).

:::{hint}
## Bonus concept: Time zones

Something we didn't cover in this lesson are time zones. In reality, time zones are important and can sometimes cause trouble in processing time series. 

For example, imagine you are searching for satellite observations of a geographic area on a specific date. If you are based on the West Coast of the US, you will be 7-8 hours behind UTC. If the satellite overpass was captured just before midnight UTC time, and you created the datetime for your search in your own time zone, you could end up missing this satellite observation because the computer interprets this time zone mismatch as an entire day's difference.

The datetime library has the ability to keep track and convert timezones, as does pandas. Here is some suggested reading to get you started with time zones in Python:
* datetime zoneinfo: https://docs.python.org/3/library/zoneinfo.html#module-zoneinfo
* time-zones in pandas: https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#time-zone-handling
* pytz: https://www.geeksforgeeks.org/python/python-pytz/
* pytz vuideo tutorial from MetPy Mondays: https://www.youtube.com/watch?v=IIriyfUmKgQ

:::{hint}
## Further reading

* [pandas time series functionality user-guide](https://pandas.pydata.org/docs/user_guide/timeseries.html)
* [date_range()](https://pandas.pydata.org/docs/reference/api/pandas.date_range.html)
* [resample()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html#pandas.DataFrame.resample)
* [groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
* [timezones with pandas](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#time-zone-handling)

MetPy Mondays videos:
* [93 Pandas and Datetime Indexes](https://youtu.be/lGZCP_3uN2U?si=MSa7lm_rl485jHhn)
* [133 Timezones](https://youtu.be/IIriyfUmKgQ?si=9iWuc_F-9-e1rwPb)
:::